In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Building Risk Score")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 12:44:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/12 12:44:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.4.0
Master: local[2]
Test: 1


In [2]:
# ==================================================
# LOAD GOLD DATA MODEL
# ==================================================

dim_building = spark.read.parquet(
    minio_path("gold/data_model/dim_building")
)

fact_311 = spark.read.parquet(
    minio_path("gold/data_model/fact_311_event")
)

fact_hpd = spark.read.parquet(
    minio_path("gold/data_model/fact_hpd_violation")
)

fact_dob = spark.read.parquet(
    minio_path("gold/data_model/fact_dob_violation")
)


print("dim_building:", dim_building.count())
print("fact_311:", fact_311.count())
print("fact_hpd:", fact_hpd.count())
print("fact_dob:", fact_dob.count())

26/09/12 12:44:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


dim_building: 197958
fact_311: 885306
fact_hpd: 927308
fact_dob: 148688


In [3]:
# ==================================================
# CHECK MAX EVENT DATES
# ==================================================

max_311_date = (
    fact_311
    .select(
        F.max("created_date").alias("max_date")
    )
    .first()["max_date"]
)

max_hpd_date = (
    fact_hpd
    .select(
        F.max("inspectiondate").alias("max_date")
    )
    .first()["max_date"]
)

max_dob_date = (
    fact_dob
    .select(
        F.max("violation_issue_date").alias("max_date")
    )
    .first()["max_date"]
)


print("311 max date:", max_311_date)
print("HPD max date:", max_hpd_date)
print("DOB max date:", max_dob_date)

311 max date: 2026-08-23 23:56:02
HPD max date: 2026-08-24 00:00:00
DOB max date: 2026-08-19 00:00:00


In [4]:
# ==================================================
# RISK AS-OF DATE
# Use the latest common date across all sources
# ==================================================

risk_as_of_date = min(
    max_311_date.date(),
    max_hpd_date.date(),
    max_dob_date.date()
)

print("Risk as-of date:", risk_as_of_date)

Risk as-of date: 2026-08-19


In [5]:
# ==================================================
# RISK AS-OF DATE
# Use the latest common date across all sources
# ==================================================

risk_as_of_date = min(
    max_311_date.date(),
    max_hpd_date.date(),
    max_dob_date.date()
)

print("Risk as-of date:", risk_as_of_date)

Risk as-of date: 2026-08-19


In [6]:
# ==================================================
# 311 BUILDING RISK FEATURES
# Non-overlapping time buckets
# ==================================================

fact_311_risk = (
    fact_311

    .filter(
        F.col("building_id").isNotNull()
    )

    # Number of days between event and risk date
    .withColumn(
        "event_age_days",
        F.datediff(
            F.lit(str(risk_as_of_date)),
            F.to_date("created_date")
        )
    )

    # Ignore events after the selected risk date
    .filter(
        F.col("event_age_days") >= 0
    )
)

In [7]:
# ==================================================
# AGGREGATE 311 FEATURES PER BUILDING
# ==================================================

features_311 = (
    fact_311_risk

    .groupBy("building_id")

    .agg(
        F.sum(
            F.when(
                F.col("event_age_days").between(0, 30),
                1
            ).otherwise(0)
        ).alias("complaints_0_30"),

        F.sum(
            F.when(
                F.col("event_age_days").between(31, 90),
                1
            ).otherwise(0)
        ).alias("complaints_31_90"),

        F.sum(
            F.when(
                F.col("event_age_days").between(91, 365),
                1
            ).otherwise(0)
        ).alias("complaints_91_365"),

        F.sum(
            F.when(
                F.col("event_age_days").between(0, 365),
                1
            ).otherwise(0)
        ).alias("complaints_365")
    )
)

In [8]:
print(
    "Buildings with 311 features:",
    features_311.count()
)

features_311.orderBy(
    F.desc("complaints_365")
).show(
    10,
    truncate=False
)

Buildings with 311 features: 46996


+-----------+---------------+----------------+-----------------+--------------+
|building_id|complaints_0_30|complaints_31_90|complaints_91_365|complaints_365|
+-----------+---------------+----------------+-----------------+--------------+
|BLD:4009048|1114           |1190            |558              |2862          |
|BLD:2059487|0              |0               |2405             |2405          |
|BLD:2017457|10             |2               |1774             |1786          |
|BLD:2003187|21             |33              |1500             |1554          |
|BLD:3094768|87             |442             |813              |1342          |
|BLD:2071915|4              |19              |1285             |1308          |
|BLD:1055063|4              |11              |1199             |1214          |
|BLD:2000000|103            |345             |766              |1214          |
|BLD:1005755|35             |40              |1132             |1207          |
|BLD:4209577|290            |23         

In [9]:
# ==================================================
# HPD STATUS + CLASS PROFILING
# ==================================================

print("HPD CLASSES")

(
    fact_hpd
    .groupBy("class")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)


print("HPD VIOLATION STATUS")

(
    fact_hpd
    .groupBy("violationstatus")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)


print("HPD CURRENT STATUS")

(
    fact_hpd
    .groupBy("currentstatus")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)

HPD CLASSES
+-----+------+
|class|count |
+-----+------+
|B    |370033|
|C    |273694|
|A    |214695|
|I    |68886 |
+-----+------+

HPD VIOLATION STATUS
+---------------+------+
|violationstatus|count |
+---------------+------+
|OPEN           |511836|
|CLOSE          |415472|
+---------------+------+

HPD CURRENT STATUS
+----------------------------------------+------+
|currentstatus                           |count |
+----------------------------------------+------+
|NOV SENT OUT                            |313895|
|VIOLATION DISMISSED                     |222941|
|VIOLATION CLOSED                        |191816|
|INFO NOV SENT OUT                       |45101 |
|FIRST NO ACCESS TO RE- INSPECT VIOLATION|32802 |
|NOT COMPLIED WITH                       |30412 |
|NOTICE OF ISSUANCE SENT TO TENANT       |29220 |
|CIV14 MAILED                            |14094 |
|VIOLATION WILL BE REINSPECTED           |11883 |
|INVALID CERTIFICATION                   |8501  |
|NOV CERTIFIED ON TIME    

In [10]:
# ==================================================
# PREPARE HPD FOR BUILDING RISK
# ==================================================

fact_hpd_risk = (
    fact_hpd

    # Building Risk requires a resolved building
    .filter(
        F.col("building_id").isNotNull()
    )

    # Calculate age of each violation relative to Risk As-Of Date
    .withColumn(
        "event_age_days",
        F.datediff(
            F.lit(str(risk_as_of_date)),
            F.to_date("inspectiondate")
        )
    )

    # Do not use violations created after the Risk As-Of Date
    .filter(
        F.col("event_age_days") >= 0
    )
)

print(
    "HPD rows used for risk:",
    fact_hpd_risk.count()
)

HPD rows used for risk: 917513


In [11]:
# ==================================================
# HPD FEATURES PER BUILDING
# ==================================================

features_hpd = (
    fact_hpd_risk

    .groupBy("building_id")

    .agg(
        # ------------------------------------------
        # ACTIVE VIOLATIONS BY HPD CLASS
        # ------------------------------------------

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "A"),
                1
            ).otherwise(0)
        ).alias("hpd_open_class_a"),

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "B"),
                1
            ).otherwise(0)
        ).alias("hpd_open_class_b"),

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "C"),
                1
            ).otherwise(0)
        ).alias("hpd_open_class_c"),

        F.sum(
            F.when(
                (F.col("violationstatus") == "OPEN")
                & (F.col("class") == "I"),
                1
            ).otherwise(0)
        ).alias("hpd_open_class_i"),

        # Total active violations
        F.sum(
            F.when(
                F.col("violationstatus") == "OPEN",
                1
            ).otherwise(0)
        ).alias("hpd_active_violations"),

        # ------------------------------------------
        # NON-OVERLAPPING RECENCY BUCKETS
        # ------------------------------------------

        F.sum(
            F.when(
                F.col("event_age_days").between(0, 30),
                1
            ).otherwise(0)
        ).alias("hpd_0_30"),

        F.sum(
            F.when(
                F.col("event_age_days").between(31, 90),
                1
            ).otherwise(0)
        ).alias("hpd_31_90"),

        F.sum(
            F.when(
                F.col("event_age_days").between(91, 365),
                1
            ).otherwise(0)
        ).alias("hpd_91_365"),

        # Analytics / validation only
        F.sum(
            F.when(
                F.col("event_age_days").between(0, 365),
                1
            ).otherwise(0)
        ).alias("hpd_365")
    )
)

In [12]:
print(
    "Buildings with HPD features:",
    features_hpd.count()
)

features_hpd.orderBy(
    F.desc("hpd_active_violations")
).show(
    10,
    truncate=False
)

Buildings with HPD features: 124710


+-----------+----------------+----------------+----------------+----------------+---------------------+--------+---------+----------+-------+
|building_id|hpd_open_class_a|hpd_open_class_b|hpd_open_class_c|hpd_open_class_i|hpd_active_violations|hpd_0_30|hpd_31_90|hpd_91_365|hpd_365|
+-----------+----------------+----------------+----------------+----------------+---------------------+--------+---------+----------+-------+
|BLD:2050000|70              |484             |152             |0               |706                  |32      |51       |743       |826    |
|BLD:2004223|60              |370             |149             |1               |580                  |22      |67       |895       |984    |
|BLD:1079927|88              |377             |103             |0               |568                  |169     |95       |677       |941    |
|BLD:2007890|76              |324             |81              |0               |481                  |67      |90       |586       |743    |
|BLD:2

In [13]:
# ==================================================
# DOB STATUS PROFILING
# ==================================================

print("DOB VIOLATION STATUS")

(
    fact_dob
    .groupBy("violation_status")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)

DOB VIOLATION STATUS
+--------------------------+------+
|violation_status          |count |
+--------------------------+------+
|ACTIVE                    |127878|
|DISMISSED                 |17309 |
|DISPUTED SUCCESSFULLY     |2543  |
|WAIVED - PENDING DISMISSAL|412   |
|null                      |396   |
|PENDING DISMISSAL         |77    |
|CURED                     |73    |
+--------------------------+------+



In [14]:
# ==================================================
# PREPARE DOB FOR BUILDING RISK
# ==================================================

fact_dob_risk = (
    fact_dob

    # Building Risk requires a resolved building
    .filter(
        F.col("building_id").isNotNull()
    )

    # Age of violation relative to Risk As-Of Date
    .withColumn(
        "event_age_days",
        F.datediff(
            F.lit(str(risk_as_of_date)),
            F.to_date("violation_issue_date")
        )
    )

    # Ignore events after the selected Risk As-Of Date
    .filter(
        F.col("event_age_days") >= 0
    )
)

print(
    "DOB rows used for risk:",
    fact_dob_risk.count()
)

DOB rows used for risk: 148688


In [15]:
# ==================================================
# DOB FEATURES PER BUILDING
# ==================================================

features_dob = (
    fact_dob_risk

    .groupBy("building_id")

    .agg(
        # ------------------------------------------
        # ACTIVE VIOLATIONS
        # ------------------------------------------

        F.sum(
            F.when(
                F.col("violation_status") == "ACTIVE",
                1
            ).otherwise(0)
        ).alias("dob_active_violations"),

        # ------------------------------------------
        # NON-OVERLAPPING RECENCY BUCKETS
        # ------------------------------------------

        F.sum(
            F.when(
                F.col("event_age_days").between(0, 30),
                1
            ).otherwise(0)
        ).alias("dob_0_30"),

        F.sum(
            F.when(
                F.col("event_age_days").between(31, 90),
                1
            ).otherwise(0)
        ).alias("dob_31_90"),

        F.sum(
            F.when(
                F.col("event_age_days").between(91, 365),
                1
            ).otherwise(0)
        ).alias("dob_91_365"),

        # Analytics / validation only
        F.sum(
            F.when(
                F.col("event_age_days").between(0, 365),
                1
            ).otherwise(0)
        ).alias("dob_365")
    )
)

In [16]:
print(
    "Buildings with DOB features:",
    features_dob.count()
)

features_dob.orderBy(
    F.desc("dob_active_violations")
).show(
    10,
    truncate=False
)

Buildings with DOB features: 106957
+-----------+---------------------+--------+---------+----------+-------+
|building_id|dob_active_violations|dob_0_30|dob_31_90|dob_91_365|dob_365|
+-----------+---------------------+--------+---------+----------+-------+
|BLD:1014453|75                   |0       |0        |75        |75     |
|BLD:3378157|62                   |0       |0        |62        |62     |
|BLD:1001394|62                   |0       |0        |62        |62     |
|BLD:1086514|56                   |0       |0        |56        |56     |
|BLD:3347532|47                   |0       |0        |47        |47     |
|BLD:1042468|47                   |0       |0        |47        |47     |
|BLD:1066406|46                   |0       |0        |46        |46     |
|BLD:1081661|43                   |0       |0        |43        |43     |
|BLD:1035381|43                   |0       |0        |43        |43     |
|BLD:2096863|43                   |0       |0        |43        |43     |
+-

In [17]:
# ==================================================
# CHECK DIM_BUILDING STRUCTURE
# ==================================================

dim_building.printSchema()

root
 |-- building_id: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- resolved_bbl: string (nullable = true)
 |-- current_bbl: string (nullable = true)
 |-- current_address: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- bbl_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- address_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- identity_status: string (nullable = true)
 |-- match_method: string (nullable = true)
 |-- match_confidence: string (nullable = true)
 |-- resolution_status: string (nullable = true)



In [18]:
# ==================================================
# LOAD DIM_PROPERTY
# ==================================================

dim_property = spark.read.parquet(
    minio_path("gold/data_model/dim_property")
)

print(
    "dim_property rows:",
    dim_property.count()
)

print("\nDIM_PROPERTY SCHEMA")
dim_property.printSchema()

dim_property rows: 858284

DIM_PROPERTY SCHEMA
root
 |-- property_id: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- property_address: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- landuse: string (nullable = true)
 |-- bldgclass: string (nullable = true)
 |-- yearbuilt: integer (nullable = true)
 |-- yearalter1: integer (nullable = true)
 |-- yearalter2: integer (nullable = true)
 |-- numbldgs: integer (nullable = true)
 |-- numfloors: double (nullable = true)
 |-- unitsres: integer (nullable = true)
 |-- unitstotal: integer (nullable = true)
 |-- lotarea: double (nullable = true)
 |-- bldgarea: double (nullable = true)
 |-- resarea: double (nullable = true)
 |-- comarea: double (nullable = true)
 |-- ownername: string (nullable = true)
 |-- ownertype: string (nullable = true)
 |-- snapshot_version: string (nullable = true)



In [19]:
# ==================================================
# PREPARE PROPERTY FEATURES
# ==================================================

property_features = (
    dim_property
    .select(
        "property_id",

        F.col("yearbuilt").alias("property_yearbuilt"),
        F.col("yearalter1").alias("property_yearalter1"),
        F.col("yearalter2").alias("property_yearalter2"),

        F.col("numbldgs").alias("property_numbldgs"),
        F.col("numfloors").alias("property_numfloors"),

        F.col("unitsres").alias("property_unitsres"),
        F.col("unitstotal").alias("property_unitstotal"),

        F.col("lotarea").alias("property_lotarea"),
        F.col("bldgarea").alias("property_bldgarea"),
        F.col("resarea").alias("property_resarea"),
        F.col("comarea").alias("property_comarea"),

        F.col("landuse").alias("property_landuse"),
        F.col("bldgclass").alias("property_bldgclass")
    )
)

In [20]:
# ==================================================
# ADD SOURCE PRESENCE FLAGS
# ==================================================

features_311_join = (
    features_311
    .withColumn("has_311_history", F.lit(1))
)

features_hpd_join = (
    features_hpd
    .withColumn("has_hpd_history", F.lit(1))
)

features_dob_join = (
    features_dob
    .withColumn("has_dob_history", F.lit(1))
)

In [21]:
# ==================================================
# BUILD BUILDING_RISK_FEATURES
# Grain: 1 row = 1 building_id
# ==================================================

building_risk_features = (
    dim_building

    # Property / PLUTO context
    .join(
        property_features,
        on="property_id",
        how="left"
    )

    # 311
    .join(
        features_311_join,
        on="building_id",
        how="left"
    )

    # HPD
    .join(
        features_hpd_join,
        on="building_id",
        how="left"
    )

    # DOB
    .join(
        features_dob_join,
        on="building_id",
        how="left"
    )
)

In [22]:
# ==================================================
# FILL MISSING EVENT FEATURES WITH ZERO
# ==================================================

event_feature_columns = [
    # 311
    "complaints_0_30",
    "complaints_31_90",
    "complaints_91_365",
    "complaints_365",

    # HPD
    "hpd_open_class_a",
    "hpd_open_class_b",
    "hpd_open_class_c",
    "hpd_open_class_i",
    "hpd_active_violations",
    "hpd_0_30",
    "hpd_31_90",
    "hpd_91_365",
    "hpd_365",

    # DOB
    "dob_active_violations",
    "dob_0_30",
    "dob_31_90",
    "dob_91_365",
    "dob_365",

    # Presence flags
    "has_311_history",
    "has_hpd_history",
    "has_dob_history"
]

building_risk_features = (
    building_risk_features
    .fillna(
        0,
        subset=event_feature_columns
    )
)

In [23]:
# ==================================================
# BUILDING / PROPERTY DERIVED FEATURES
# ==================================================

risk_year = risk_as_of_date.year

building_risk_features = (
    building_risk_features

    # Approximate building/property age
    .withColumn(
        "building_age",
        F.when(
            (F.col("property_yearbuilt") > 0)
            & (F.col("property_yearbuilt") <= risk_year),
            F.lit(risk_year) - F.col("property_yearbuilt")
        )
    )

    # Safe to use property units as building denominator
    # only when PLUTO says the property contains one building
    .withColumn(
        "units_normalization_eligible",
        F.when(
            (F.col("property_numbldgs") == 1)
            & (F.col("property_unitsres") > 0),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "units_for_normalization",
        F.when(
            F.col("units_normalization_eligible") == 1,
            F.col("property_unitsres")
        )
    )
)

In [24]:
print(
    "Building risk feature rows:",
    building_risk_features.count()
)

print(
    "Distinct building_id:",
    building_risk_features
    .select("building_id")
    .distinct()
    .count()
)

print(
    "With 311 history:",
    building_risk_features
    .filter(F.col("has_311_history") == 1)
    .count()
)

print(
    "With HPD history:",
    building_risk_features
    .filter(F.col("has_hpd_history") == 1)
    .count()
)

print(
    "With DOB history:",
    building_risk_features
    .filter(F.col("has_dob_history") == 1)
    .count()
)

print(
    "Eligible for units normalization:",
    building_risk_features
    .filter(F.col("units_normalization_eligible") == 1)
    .count()
)

Building risk feature rows: 197958
Distinct building_id: 197958


With 311 history: 46996


With HPD history: 124710
With DOB history: 106957
Eligible for units normalization: 122380


In [25]:
# ==================================================
# SAVE BUILDING RISK FEATURES
# ==================================================

BUILDING_RISK_FEATURES_PATH = minio_path(
    "gold/building_risk/features"
)

(
    building_risk_features
    .write
    .mode("overwrite")
    .parquet(BUILDING_RISK_FEATURES_PATH)
)

print("Building Risk Features saved successfully")
print("Path:", BUILDING_RISK_FEATURES_PATH)

26/09/12 13:44:50 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Building Risk Features saved successfully
Path: s3a://nyc-building-risk/gold/building_risk/features


In [26]:
# ==================================================
# CREATE RAW RISK SIGNALS
# ==================================================

risk_signals = (
    building_risk_features

    # ----------------------------------------------
    # 311
    # Recent complaints receive more weight
    # ----------------------------------------------
    .withColumn(
        "risk_311_raw",
        F.col("complaints_0_30") * 3
        + F.col("complaints_31_90") * 2
        + F.col("complaints_91_365")
    )

    # ----------------------------------------------
    # HPD SEVERITY
    # A = 1, B = 2, C = 4
    # Class I is kept separately and is not
    # treated as a severity level.
    # ----------------------------------------------
    .withColumn(
        "risk_hpd_severity_raw",
        F.col("hpd_open_class_a")
        + F.col("hpd_open_class_b") * 2
        + F.col("hpd_open_class_c") * 4
    )

    # ----------------------------------------------
    # HPD RECENCY
    # ----------------------------------------------
    .withColumn(
        "risk_hpd_recency_raw",
        F.col("hpd_0_30") * 3
        + F.col("hpd_31_90") * 2
        + F.col("hpd_91_365")
    )

    # ----------------------------------------------
    # DOB ACTIVE
    # ----------------------------------------------
    .withColumn(
        "risk_dob_active_raw",
        F.col("dob_active_violations")
    )

    # ----------------------------------------------
    # DOB RECENCY
    # ----------------------------------------------
    .withColumn(
        "risk_dob_recency_raw",
        F.col("dob_0_30") * 3
        + F.col("dob_31_90") * 2
        + F.col("dob_91_365")
    )
)

In [27]:
# ==================================================
# PROFILE RAW RISK SIGNALS
# ==================================================

risk_columns = [
    "risk_311_raw",
    "risk_hpd_severity_raw",
    "risk_hpd_recency_raw",
    "risk_dob_active_raw",
    "risk_dob_recency_raw",
    "building_age"
]

for column_name in risk_columns:

    quantiles = risk_signals.approxQuantile(
        column_name,
        [0.50, 0.90, 0.95, 0.99],
        0.01
    )

    print(
        column_name,
        "P50 =", quantiles[0],
        "P90 =", quantiles[1],
        "P95 =", quantiles[2],
        "P99 =", quantiles[3]
    )

risk_311_raw P50 = 0.0 P90 = 9.0 P95 = 26.0 P99 = 6280.0


risk_hpd_severity_raw P50 = 0.0 P90 = 8.0 P95 = 25.0 P99 = 1646.0
risk_hpd_recency_raw P50 = 1.0 P90 = 9.0 P95 = 26.0 P99 = 1398.0
risk_dob_active_raw P50 = 0.0 P90 = 1.0 P95 = 2.0 P99 = 75.0
risk_dob_recency_raw P50 = 1.0 P90 = 2.0 P95 = 3.0 P99 = 82.0
building_age P50 = 96.0 P90 = 125.0 P95 = 127.0 P99 = 365.0


In [28]:
# ==================================================
# MORE PRECISE RISK DISTRIBUTION PROFILE
# ==================================================

risk_columns = [
    "risk_311_raw",
    "risk_hpd_severity_raw",
    "risk_hpd_recency_raw",
    "risk_dob_active_raw",
    "risk_dob_recency_raw",
    "building_age"
]

percentiles = [
    0.50,
    0.90,
    0.95,
    0.975,
    0.99,
    0.995
]

for column_name in risk_columns:

    quantiles = risk_signals.approxQuantile(
        column_name,
        percentiles,
        0.001
    )

    print(
        "\n",
        column_name,
        "\nP50  =", quantiles[0],
        "\nP90  =", quantiles[1],
        "\nP95  =", quantiles[2],
        "\nP97.5=", quantiles[3],
        "\nP99  =", quantiles[4],
        "\nP99.5=", quantiles[5]
    )


 risk_311_raw 
P50  = 0.0 
P90  = 10.0 
P95  = 26.0 
P97.5= 50.0 
P99  = 96.0 
P99.5= 131.0



 risk_hpd_severity_raw 
P50  = 0.0 
P90  = 8.0 
P95  = 25.0 
P97.5= 51.0 
P99  = 102.0 
P99.5= 143.0

 risk_hpd_recency_raw 
P50  = 1.0 
P90  = 10.0 
P95  = 26.0 
P97.5= 52.0 
P99  = 104.0 
P99.5= 145.0

 risk_dob_active_raw 
P50  = 0.0 
P90  = 1.0 
P95  = 2.0 
P97.5= 3.0 
P99  = 4.0 
P99.5= 6.0

 risk_dob_recency_raw 
P50  = 1.0 
P90  = 2.0 
P95  = 3.0 
P97.5= 4.0 
P99  = 6.0 
P99.5= 7.0

 building_age 
P50  = 96.0 
P90  = 125.0 
P95  = 127.0 
P97.5= 140.0 
P99  = 160.0 
P99.5= 174.0


In [29]:
# ==================================================
# ROBUST NORMALIZATION
# ==================================================

CAP_311 = 96.0
CAP_HPD_SEVERITY = 102.0
CAP_HPD_RECENCY = 104.0
CAP_DOB_ACTIVE = 4.0
CAP_DOB_RECENCY = 6.0
CAP_BUILDING_AGE = 160.0


def log_normalize(column_name, cap_value):
    return (
        F.log1p(
            F.least(
                F.col(column_name),
                F.lit(cap_value)
            )
        )
        / F.log1p(F.lit(cap_value))
        * 100
    )


risk_normalized = (
    risk_signals

    .withColumn(
        "score_311",
        log_normalize(
            "risk_311_raw",
            CAP_311
        )
    )

    .withColumn(
        "score_hpd_severity",
        log_normalize(
            "risk_hpd_severity_raw",
            CAP_HPD_SEVERITY
        )
    )

    .withColumn(
        "score_hpd_recency",
        log_normalize(
            "risk_hpd_recency_raw",
            CAP_HPD_RECENCY
        )
    )

    .withColumn(
        "score_dob_active",
        log_normalize(
            "risk_dob_active_raw",
            CAP_DOB_ACTIVE
        )
    )

    .withColumn(
        "score_dob_recency",
        log_normalize(
            "risk_dob_recency_raw",
            CAP_DOB_RECENCY
        )
    )

    .withColumn(
        "score_building_age",
        F.when(
            F.col("building_age").isNotNull(),
            F.least(
                F.col("building_age"),
                F.lit(CAP_BUILDING_AGE)
            )
            / F.lit(CAP_BUILDING_AGE)
            * 100
        )
    )
)

In [30]:
risk_normalized.select(
    "building_id",
    "risk_311_raw",
    "score_311",
    "risk_hpd_severity_raw",
    "score_hpd_severity",
    "risk_dob_active_raw",
    "score_dob_active",
    "building_age",
    "score_building_age"
).orderBy(
    F.desc("score_311")
).show(
    20,
    truncate=False
)

+-----------+------------+---------+---------------------+------------------+-------------------+-----------------+------------+------------------+
|building_id|risk_311_raw|score_311|risk_hpd_severity_raw|score_hpd_severity|risk_dob_active_raw|score_dob_active |building_age|score_building_age|
+-----------+------------+---------+---------------------+------------------+-------------------+-----------------+------------+------------------+
|BLD:1030143|97          |100.0    |2                    |23.703916484828927|1                  |43.06765580733931|135         |84.375            |
|BLD:1034191|440         |100.0    |197                  |100.0             |4                  |100.0            |86          |53.75             |
|BLD:1051901|162         |100.0    |250                  |100.0             |0                  |0.0              |116         |72.5              |
|BLD:1053802|119         |100.0    |127                  |100.0             |2                  |68.260619448598

In [31]:
# ==================================================
# FINAL BUILDING RISK SCORE
# ==================================================

risk_scored = (
    risk_normalized

    # --------------------------------------------------
    # Handle missing building age
    # Median age = 96 years
    # Normalized: 96 / 160 * 100 = 60
    # --------------------------------------------------
    .withColumn(
        "age_imputed_flag",
        F.when(
            F.col("score_building_age").isNull(),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "score_building_age_final",
        F.coalesce(
            F.col("score_building_age"),
            F.lit(60.0)
        )
    )

    # --------------------------------------------------
    # COMPONENT CONTRIBUTIONS
    # --------------------------------------------------
    .withColumn(
        "risk_points_311",
        F.col("score_311") * 0.25
    )

    .withColumn(
        "risk_points_hpd_severity",
        F.col("score_hpd_severity") * 0.28
    )

    .withColumn(
        "risk_points_hpd_recency",
        F.col("score_hpd_recency") * 0.12
    )

    .withColumn(
        "risk_points_dob_active",
        F.col("score_dob_active") * 0.15
    )

    .withColumn(
        "risk_points_dob_recency",
        F.col("score_dob_recency") * 0.10
    )

    .withColumn(
        "risk_points_building_age",
        F.col("score_building_age_final") * 0.10
    )

    # --------------------------------------------------
    # FINAL SCORE 0-100
    # --------------------------------------------------
    .withColumn(
        "building_risk_score",
        F.round(
            F.col("risk_points_311")
            + F.col("risk_points_hpd_severity")
            + F.col("risk_points_hpd_recency")
            + F.col("risk_points_dob_active")
            + F.col("risk_points_dob_recency")
            + F.col("risk_points_building_age"),
            2
        )
    )
)

In [32]:
risk_scored.select(
    "building_id",
    "current_address",
    "building_risk_score",

    "risk_points_311",
    "risk_points_hpd_severity",
    "risk_points_hpd_recency",
    "risk_points_dob_active",
    "risk_points_dob_recency",
    "risk_points_building_age",

    "age_imputed_flag"
).orderBy(
    F.desc("building_risk_score")
).show(
    20,
    truncate=False
)

+-----------+----------------------------+-------------------+------------------+------------------------+-----------------------+----------------------+-----------------------+------------------------+----------------+
|building_id|current_address             |building_risk_score|risk_points_311   |risk_points_hpd_severity|risk_points_hpd_recency|risk_points_dob_active|risk_points_dob_recency|risk_points_building_age|age_imputed_flag|
+-----------+----------------------------+-------------------+------------------+------------------------+-----------------------+----------------------+-----------------------+------------------------+----------------+
|BLD:1053262|2 WEST 120 STREET           |97.88              |25.0              |28.000000000000004      |12.0                   |15.0                  |10.0                   |7.875                   |0               |
|BLD:1060413|2400 ADAM C POWELL BOULEVARD|97.49              |25.0              |28.000000000000004      |11.92525711086

In [33]:
# ==================================================
# FINAL RISK SCORE DISTRIBUTION
# ==================================================

risk_score_quantiles = risk_scored.approxQuantile(
    "building_risk_score",
    [
        0.50,
        0.75,
        0.90,
        0.95,
        0.975,
        0.99,
        0.995
    ],
    0.001
)

print("Building Risk Score Distribution")
print("P50   =", risk_score_quantiles[0])
print("P75   =", risk_score_quantiles[1])
print("P90   =", risk_score_quantiles[2])
print("P95   =", risk_score_quantiles[3])
print("P97.5 =", risk_score_quantiles[4])
print("P99   =", risk_score_quantiles[5])
print("P99.5 =", risk_score_quantiles[6])

print(
    "Buildings with imputed age:",
    risk_scored
    .filter(F.col("age_imputed_flag") == 1)
    .count()
)

Building Risk Score Distribution
P50   = 15.96
P75   = 22.98
P90   = 39.55
P95   = 54.52
P97.5 = 65.25
P99   = 74.81
P99.5 = 80.08
Buildings with imputed age: 6098


In [34]:
# ==================================================
# CHECK SCORE SATURATION
# ==================================================

print(
    "311 score = 100:",
    risk_scored
    .filter(F.col("score_311") >= 99.999)
    .count()
)

print(
    "HPD severity score = 100:",
    risk_scored
    .filter(F.col("score_hpd_severity") >= 99.999)
    .count()
)

print(
    "HPD recency score = 100:",
    risk_scored
    .filter(F.col("score_hpd_recency") >= 99.999)
    .count()
)

print(
    "DOB active score = 100:",
    risk_scored
    .filter(F.col("score_dob_active") >= 99.999)
    .count()
)

print(
    "DOB recency score = 100:",
    risk_scored
    .filter(F.col("score_dob_recency") >= 99.999)
    .count()
)

311 score = 100: 2048
HPD severity score = 100: 2042


HPD recency score = 100: 2019
DOB active score = 100: 3212
DOB recency score = 100: 2137


In [35]:
# ==================================================
# ASSIGN RISK LEVEL
# ==================================================

P75_SCORE = 22.98
P95_SCORE = 54.52
P99_SCORE = 74.81


risk_scored = (
    risk_scored

    .withColumn(
        "risk_level",
        F.when(
            F.col("building_risk_score") >= P99_SCORE,
            F.lit("CRITICAL")
        )
        .when(
            F.col("building_risk_score") >= P95_SCORE,
            F.lit("HIGH")
        )
        .when(
            F.col("building_risk_score") >= P75_SCORE,
            F.lit("MEDIUM")
        )
        .otherwise(
            F.lit("LOW")
        )
    )
)

In [36]:
# ==================================================
# VALIDATE RISK LEVEL DISTRIBUTION
# ==================================================

(
    risk_scored
    .groupBy("risk_level")
    .count()
    .withColumn(
        "percent",
        F.round(
            F.col("count") / F.lit(197958) * 100,
            2
        )
    )
    .orderBy(
        F.when(F.col("risk_level") == "CRITICAL", 1)
         .when(F.col("risk_level") == "HIGH", 2)
         .when(F.col("risk_level") == "MEDIUM", 3)
         .otherwise(4)
    )
    .show(truncate=False)
)

+----------+------+-------+
|risk_level|count |percent|
+----------+------+-------+
|CRITICAL  |2017  |1.02   |
|HIGH      |7964  |4.02   |
|MEDIUM    |39655 |20.03  |
|LOW       |148322|74.93  |
+----------+------+-------+



In [37]:
# ==================================================
# FINAL BUILDING RISK OUTPUT
# Grain: 1 row = 1 building_id
# ==================================================

building_risk_score = (
    risk_scored

    .select(
        # -----------------------------
        # Identity
        # -----------------------------
        "building_id",
        "bin",
        "property_id",
        "current_address",
        "borough",
        "latitude",
        "longitude",

        # -----------------------------
        # Final Risk
        # -----------------------------
        "building_risk_score",
        "risk_level",

        # -----------------------------
        # Component Scores
        # -----------------------------
        "score_311",
        "score_hpd_severity",
        "score_hpd_recency",
        "score_dob_active",
        "score_dob_recency",
        "score_building_age_final",

        # -----------------------------
        # Weighted Contributions
        # -----------------------------
        "risk_points_311",
        "risk_points_hpd_severity",
        "risk_points_hpd_recency",
        "risk_points_dob_active",
        "risk_points_dob_recency",
        "risk_points_building_age",

        # -----------------------------
        # Raw Signals
        # -----------------------------
        "risk_311_raw",
        "risk_hpd_severity_raw",
        "risk_hpd_recency_raw",
        "risk_dob_active_raw",
        "risk_dob_recency_raw",

        # -----------------------------
        # Main Features
        # -----------------------------
        "complaints_0_30",
        "complaints_31_90",
        "complaints_91_365",

        "hpd_open_class_a",
        "hpd_open_class_b",
        "hpd_open_class_c",
        "hpd_open_class_i",
        "hpd_active_violations",

        "dob_active_violations",

        # -----------------------------
        # Building Context
        # -----------------------------
        "property_yearbuilt",
        "building_age",
        "property_numbldgs",
        "property_unitsres",
        "property_unitstotal",

        "age_imputed_flag",
        "units_normalization_eligible"
    )
)

In [38]:
print(
    "Risk rows:",
    building_risk_score.count()
)

print(
    "Distinct building_id:",
    building_risk_score
    .select("building_id")
    .distinct()
    .count()
)

building_risk_score.select(
    F.min("building_risk_score").alias("min_score"),
    F.max("building_risk_score").alias("max_score"),
    F.avg("building_risk_score").alias("avg_score")
).show()

Risk rows: 197958
Distinct building_id: 197958


+---------+---------+-----------------+
|min_score|max_score|        avg_score|
+---------+---------+-----------------+
|     0.69|    97.88|19.84235459036719|
+---------+---------+-----------------+



In [39]:
# ==================================================
# SAVE BUILDING RISK SCORE
# ==================================================

BUILDING_RISK_SCORE_PATH = minio_path(
    "gold/building_risk/building_risk_score"
)

(
    building_risk_score
    .write
    .mode("overwrite")
    .parquet(BUILDING_RISK_SCORE_PATH)
)

print("Building Risk Score saved successfully")
print("Path:", BUILDING_RISK_SCORE_PATH)

Building Risk Score saved successfully
Path: s3a://nyc-building-risk/gold/building_risk/building_risk_score
